# PO3 Trader — Pattern Visualization & Backtest

This notebook lets you:
- Fetch historical BTC/USDT M1 candles
- Define your setup (Direction, Range, Target)
- Run the backtester to find all PO3 patterns
- Visualize each pattern on a candlestick chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

from data_feed import get_exchange, fetch_candles
from backtester import scan_pattern, BACKTEST_CANDLES
from config import SYMBOL, TIMEFRAME_LTF

## 1. Define Setup

In [ ]:
setup = {
    "direction"   : "LONG",       # "LONG" or "SHORT"
    "range_high"  : 70179.53,
    "range_low"   : 69081.51,
    "target_level": 71500.00,
    "range_size"  : round(70179.53 - 69081.51, 2),
}

print(f"Direction    : {setup['direction']}")
print(f"Range High   : {setup['range_high']}")
print(f"Range Low    : {setup['range_low']}")
print(f"Range Size   : ${setup['range_size']}")
print(f"Target Level : {setup['target_level']}")

## 2. Fetch Historical Data

In [ ]:
exchange = get_exchange()
df = fetch_candles(exchange, SYMBOL, TIMEFRAME_LTF, BACKTEST_CANDLES)
print(f"Loaded {len(df)} M1 candles")
print(f"From : {df.index[0]}")
print(f"To   : {df.index[-1]}")
df.tail(3)

## 3. Run Backtester

In [ ]:
results = scan_pattern(df, setup)
print(f"Patterns found: {len(results)}")
for i, r in enumerate(results, 1):
    print(f"\n  Pattern #{i}")
    print(f"  Confirmed : {r['confirm_time']}")
    print(f"  Entry     : {r['entry_price']:.2f}")
    print(f"  SL        : {r['sl']:.2f}")
    print(f"  TP        : {r['tp']:.2f}")
    print(f"  R/R       : 1:{r['rr']}")

## 4. Candlestick Chart with Pattern Markers

In [ ]:
def plot_pattern(df, result, setup, window=60):
    """Plot a candlestick window around a confirmed pattern."""
    confirm_time = result["confirm_time"]
    idx = df.index.get_loc(confirm_time)
    start = max(0, idx - window)
    end   = min(len(df), idx + 10)
    sub   = df.iloc[start:end].copy()

    fig, ax = plt.subplots(figsize=(16, 6))
    x = range(len(sub))

    # Draw candles
    for i, (ts, row) in enumerate(sub.iterrows()):
        color = "#26a69a" if row["close"] >= row["open"] else "#ef5350"
        ax.plot([i, i], [row["low"], row["high"]], color=color, linewidth=0.8)
        ax.add_patch(plt.Rectangle(
            (i - 0.3, min(row["open"], row["close"])),
            0.6, abs(row["close"] - row["open"]),
            color=color
        ))

    # Range High / Low
    ax.axhline(setup["range_high"], color="orange", linestyle="--", linewidth=1, label="Range High")
    ax.axhline(setup["range_low"],  color="orange", linestyle="--", linewidth=1, label="Range Low")

    # SL / TP / Entry
    ax.axhline(result["entry_price"], color="white",  linestyle=":",  linewidth=1, label=f"Entry {result['entry_price']:.2f}")
    ax.axhline(result["sl"],          color="#ef5350", linestyle="-.", linewidth=1, label=f"SL {result['sl']:.2f}")
    ax.axhline(result["tp"],          color="#26a69a", linestyle="-.", linewidth=1, label=f"TP {result['tp']:.2f}")

    # F1 / F2 markers
    if result["f1_time"] in sub.index:
        f1_x = sub.index.get_loc(result["f1_time"])
        ax.annotate("F1", xy=(f1_x, result["f1"]), color="yellow", fontsize=9,
                    xytext=(f1_x, result["f1"] * 0.9995),
                    arrowprops=dict(arrowstyle="->", color="yellow"))

    if result["f2_time"] in sub.index:
        f2_x = sub.index.get_loc(result["f2_time"])
        ax.annotate("F2", xy=(f2_x, result["f2"]), color="cyan", fontsize=9,
                    xytext=(f2_x, result["f2"] * 1.0005),
                    arrowprops=dict(arrowstyle="->", color="cyan"))

    ax.set_facecolor("#131722")
    fig.patch.set_facecolor("#131722")
    ax.tick_params(colors="white")
    ax.yaxis.label.set_color("white")
    ax.set_title(f"PO3 Pattern — {setup['direction']} | Confirmed: {confirm_time}", color="white")
    ax.set_xticks(range(0, len(sub), max(1, len(sub)//10)))
    ax.set_xticklabels([str(sub.index[i])[-8:-3] for i in range(0, len(sub), max(1, len(sub)//10))],
                       rotation=30, color="white", fontsize=7)
    ax.legend(facecolor="#1e222d", labelcolor="white", fontsize=8)
    plt.tight_layout()
    plt.show()


for r in results:
    plot_pattern(df, r, setup)

## 5. Summary Statistics

In [ ]:
if results:
    rr_values = [r["rr"] for r in results if r["rr"] is not None]
    risks     = [r["risk"] for r in results]
    rewards   = [r["reward"] for r in results]

    print(f"Total patterns   : {len(results)}")
    print(f"Avg R/R          : 1:{sum(rr_values)/len(rr_values):.2f}")
    print(f"Avg Risk ($)     : {sum(risks)/len(risks):.2f}")
    print(f"Avg Reward ($)   : {sum(rewards)/len(rewards):.2f}")
    print(f"Best R/R         : 1:{max(rr_values):.2f}")
    print(f"Worst R/R        : 1:{min(rr_values):.2f}")
else:
    print("No patterns found in this window.")